In [309]:
import time
import random
import string
from collections import defaultdict

In [253]:
class Addressable:
    def __init__(self):
        self.address = '0x' + ''.join(random.choices('0123456789abcdef', k=40))

In [254]:
class User(Addressable): pass

In [255]:
vitalik = User()
satoshi = User()
merkle  = User()
shannon = User()
shamir  = User()

In [256]:
class Token(Addressable):
    def __init__(self, name, symbol): # no need for decimals
        super().__init__()
        self.name   = name
        self.symbol = symbol
        
        self.balances    = {}
        self.totalSupply = 0

    def transferFrom(self, fr, to, amount):
        assert self.balances[fr.address] >= amount
        self.balances[fr.address] -= amount
        self.balances[to.address]  = 0
        self.balances[to.address] += amount

    def _mint(self, to, amount):
        self.balances[to.address]  = 0
        self.balances[to.address] += amount
        self.totalSupply          += amount

    def _burn(self, fr, amount):
        self.balances[fr.address]  = 0
        self.balances[fr.address] -= amount
        self.totalSupply          -= amount

    def balanceOf(self, user): return self.balances[user.address]

In [257]:
DYAD = Token("DYAD Stable",   "DYAD")
USDC = Token("USD Coin",      "USDC")
WETH = Token("Wrapped Ether", "WETH")
WBTC = Token("Wrapped BTC",   "WBTC")
USDT = Token("Tether USD",    "USDT")

In [258]:
DYAD._mint(vitalik, 100)
DYAD.transferFrom(vitalik, satoshi, 20)

DYAD.balanceOf(vitalik)

80

In [259]:
class Oracle(Addressable):
    def __init__(self, asset0, asset1, price): 
        self.asset0, self.asset1 = asset0, asset1
        self.price = price
    def set_price(self, price): self.price = price
    def __str__(self): 
        return f"{self.asset0.symbol}/{self.asset1.symbol} {self.price}"

In [260]:
ETH2USD = Oracle(WETH, USDC, 2615.93)
ETH2BTC = Oracle(WETH, WBTC, 0.039)

print(str(ETH2USD))
print(str(ETH2BTC))

WETH/USDC 2615.93
WETH/WBTC 0.039


In [313]:
class MicroLend(Addressable):
    def __init__(
        self,
        lend_asset,
        collat_asset,
        borrow_asset,
        min_collat_ratio
    ):
        self.lend_asset        = lend_asset
        self.collat_asset      = collat_asset
        self.borrow_asset      = borrow_asset
        self.min_collat_ratio  = min_collat_ratio
        self.balances          = defaultdict(lambda: defaultdict(int))
        self.totals            = defaultdict(int)
        
        self.totalInterestPerLend   = 0
        self.totalInterestPerBorrow = 0
        self.lastUpdateLend         = get_time()
        self.lastUpdateBorrow       = get_time()
        self.interestPerLendPaid    = defaultdict(int)
        self.interestPerBorrowPaid  = defaultdict(int)
        self.totalLendReward        = defaultdict(int)

    def get_time(): return int(time.time())

    def apr_to_spr(x): return x / (356 * 24 * 60 * 60)

    def reward_per_token(self):
        totalLend = self.totals[self.lend_asset.address]
        if totalLend == 0: return totalInterestPerLend

        spr = apr_to_spr(self.lend_apr)
        dt  = get_time() - self.lastUpdate

        return totalInterestPerLend + (spr * dt) / totalLend

    def earned(self, user):
        balance = self.balances[user.address][self.lend_asset.address]
        d_rewardPerToken = self.reward_per_token - self.interestPerLendPaid[user.address]
        return self.totalLendReward[user.address] + balance * d_rewardPerToken

    def deposit(self, user, token, amount):
        self.balances[user.address][token.address] += amount
        self.totals[token.address]                 += amount

    def withdraw(self, user, token, amount):
        self.balances[user.address][token.address] -= amount
        self.totals[token.address]                 -= amount

    def borrow(self, user, amount):
        self.balances[user.address][self.borrow_asset.address] += amount
        self.totals[self.borrow_asset.address]                 += amount

    def repay(self, user, amount): pass

    def collat_ratio(self, user): 
        borrowed = self.balances[user.address][self.borrow_asset.address]
        if borrowed == 0: return (1<<256) - 1 # a very large uint 
        return self.balances[user.address][self.collat_asset.address] / borrowed

    def utilization(self):
        lend = self.totals[self.lend_asset.address]
        if lend == 0: return 0
        return self.totals[self.borrow_asset.address] / lend

    def borrow_apr(self): return self.utilization() * 0.056

    def lend_apr(self):
        return self.utilization() * self.borrow_apr() * 0.9

    def stats(self, user):
        lend   = self.balances[user.address][self.lend_asset.address]
        collat = self.balances[user.address][self.collat_asset.address]
        borrow = self.balances[user.address][self.borrow_asset.address]
        cr     = round(self.collat_ratio(user)*100, 2)
        print(
            f"{'Lend:':<14} {lend} {self.lend_asset.symbol}\n"
            f"{'Collateral:':<14} {collat}\n"
            f"{'Borrowed:':<14} {borrow} {self.borrow_asset.symbol}\n"
            f"{'Collat Ratio:':<14} {cr}%\n"
            "\n"
            f"{'Utilization:':<14} {self.utilization()*100}%\n"
            f"{'Lend APR:':<14} {self.lend_apr()*100}%\n"
            f"{'Borrow APR:':<14} {self.borrow_apr()*100}%"
        )

In [306]:
mL = MicroLend(
    WETH, 
    USDC,
    DYAD,
    1.5   # 150%
)

In [307]:
mL.deposit(vitalik, WETH, 1_000)

mL.deposit(vitalik, USDC, 200)
mL.borrow (vitalik, 130)

In [308]:
mL.stats(vitalik)

Lend:          1000 WETH
Collateral:    200
Borrowed:      130 DYAD
Collat Ratio:  153.85%

Utilization:   13.0%
Lend APR:      0.085176%
Borrow APR:    0.728%


In [311]:
int(time.time())

1729158438